# Healthcare Patient Outcome Analysis (30-day Readmission)
**Tools:** SQL (SQLite), Python (Pandas, NumPy, SciPy, Statsmodels, Matplotlib, Plotly), Excel, Power BI

**What this project does:** It builds a hospital discharge dataset, extracts cohorts with SQL,
cleans and prepares the data in Python, uses correlation analysis and logistic regression to find
the factors linked to 30-day readmission, estimates how much readmission could drop with
interventions, and exports files for Excel and Power BI.

**Note:** The data is synthetic (generated by code with a fixed random seed). It is not real
patient data. The intervention numbers are model-based estimates, not proven clinical results.

## 0. Setup

In [ ]:
import os
import sqlite3
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    import statsmodels.formula.api as smf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    import statsmodels.formula.api as smf

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (9, 4.5)
OUT = "outputs_healthcare"
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(7)  # fixed seed = same results every run

## 1. Create the patient dataset (no download needed)
Each row is one hospital discharge. Whether the patient is readmitted within 30 days is
generated from clinical factors (age, prior admissions, comorbidities, diagnosis, follow-up
visit and so on) plus random chance, so the analysis has real patterns to discover.

In [ ]:
N = 8000
diag_list = ["Diabetes", "Heart Failure", "COPD", "Pneumonia", "Hypertension", "Kidney Disease"]
diag_p = [0.25, 0.18, 0.15, 0.14, 0.18, 0.10]
ins_list = ["Medicare", "Private", "Medicaid", "Self-pay"]

age = np.clip(rng.normal(62, 16, N), 18, 95).round().astype(int)
gender = rng.choice(["Male", "Female"], N)
diagnosis = rng.choice(diag_list, N, p=diag_p)
insurance = np.where(age >= 65,
                     rng.choice(ins_list, N, p=[.75, .15, .08, .02]),
                     rng.choice(ins_list, N, p=[.10, .55, .27, .08]))

heavy_dx = np.isin(diagnosis, ["Heart Failure", "COPD"])
prior_adm = np.clip(rng.poisson(0.9 + heavy_dx * 0.8 + (age >= 75) * 0.3), 0, 10)
comorb = np.clip(rng.poisson(1.0 + age / 50), 0, 8)
meds = np.clip(np.round(rng.normal(7 + 1.3 * comorb, 3)), 1, 30).astype(int)
los = np.clip(np.round(rng.gamma(2.2, 1.7, N) + 0.4 * comorb
                       + np.isin(diagnosis, ["Heart Failure", "Pneumonia"]) * 1.2), 1, 30).astype(int)
labs = np.clip(np.round(rng.normal(35 + 2.5 * los, 12)), 1, 120).astype(int)

u = rng.random(N)
p_nursing = np.where(age >= 75, 0.25, 0.05)
p_home_health = np.where(age >= 65, 0.25, 0.12)
disposition = np.where(u < p_nursing, "Nursing Facility",
                       np.where(u < p_nursing + p_home_health, "Home Health", "Home"))
followup = (rng.random(N) < np.where(disposition == "Home Health", 0.70, 0.50)).astype(int)
bmi = np.round(np.clip(rng.normal(28, 5.5, N), 15, 55), 1)
sbp = np.round(np.clip(rng.normal(130, 17, N), 80, 220))

diag_eff = {"Heart Failure": 0.6, "COPD": 0.45, "Kidney Disease": 0.4,
            "Diabetes": 0.2, "Pneumonia": 0.1, "Hypertension": 0.0}
ins_eff = {"Self-pay": 0.3, "Medicaid": 0.15, "Medicare": 0.0, "Private": 0.0}
disp_eff = {"Nursing Facility": 0.3, "Home Health": -0.1, "Home": 0.0}
z = (-3.0 + 0.02 * (age - 60) + 0.35 * prior_adm + 0.25 * comorb + 0.04 * los
     + 0.02 * (meds - 8)
     + pd.Series(diagnosis).map(diag_eff).values
     + pd.Series(insurance).map(ins_eff).values
     + pd.Series(disposition).map(disp_eff).values
     - 0.7 * followup)
readmitted = (rng.random(N) < 1 / (1 + np.exp(-z))).astype(int)

admission_date = pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 1096, N), unit="D")

df = pd.DataFrame({
    "patient_id": [f"P{100000 + i}" for i in range(N)],
    "admission_date": admission_date,
    "age": age, "gender": gender, "diagnosis": diagnosis, "insurance": insurance,
    "prior_admissions": prior_adm, "comorbidity_count": comorb,
    "num_medications": meds, "length_of_stay": los, "num_lab_procedures": labs,
    "discharge_disposition": disposition, "followup_7d": followup,
    "bmi": bmi, "systolic_bp": sbp, "readmitted": readmitted,
})

# Make the raw data a little messy on purpose, like real records
df.loc[rng.random(N) < 0.04, "bmi"] = np.nan
df.loc[rng.random(N) < 0.03, "systolic_bp"] = np.nan
df = pd.concat([df, df.sample(40, random_state=1)], ignore_index=True)  # duplicate rows
print("Raw shape:", df.shape)
df.head()

## 2. Data cleaning and preprocessing

In [ ]:
print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0], "\n")
print("Duplicate patient ids:", df["patient_id"].duplicated().sum())

df = df.drop_duplicates("patient_id").reset_index(drop=True)
for col in ["bmi", "systolic_bp"]:
    df[col] = df[col].fillna(df[col].median())  # median is safe against outliers

df["age_group"] = pd.cut(df["age"], [17, 39, 54, 64, 74, 120],
                         labels=["18-39", "40-54", "55-64", "65-74", "75+"])
df["los_group"] = pd.cut(df["length_of_stay"], [0, 3, 6, 10, 100],
                         labels=["1-3 days", "4-6 days", "7-10 days", "11+ days"])
df["prior_group"] = pd.cut(df["prior_admissions"], [-1, 0, 1, 2, 3, 100],
                           labels=["0", "1", "2", "3", "4+"])
df["followup_label"] = np.where(df["followup_7d"] == 1, "Follow-up within 7 days", "No follow-up")
df["admission_year"] = df["admission_date"].dt.year

print("Clean shape:", df.shape)
print(f"Overall 30-day readmission rate: {df['readmitted'].mean():.2%}")

## 3. SQL data extraction (SQLite, runs inside Python)

In [ ]:
conn = sqlite3.connect(":memory:")
df_sql = df.copy()
for col in ["age_group", "los_group", "prior_group"]:
    df_sql[col] = df_sql[col].astype(str)
df_sql["admission_date"] = df_sql["admission_date"].dt.strftime("%Y-%m-%d")
df_sql.to_sql("patients", conn, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql_query(sql, conn)

# 3.1 Readmission rate by diagnosis
print(q("""
SELECT diagnosis,
       COUNT(*)                          AS patients,
       SUM(readmitted)                   AS readmissions,
       ROUND(100.0 * AVG(readmitted), 2) AS readmit_rate_pct,
       ROUND(AVG(length_of_stay), 2)     AS avg_los
FROM patients
GROUP BY diagnosis
ORDER BY readmit_rate_pct DESC
"""))

In [ ]:
# 3.2 Does a follow-up visit within 7 days matter?
print(q("""
SELECT followup_7d,
       COUNT(*) AS patients,
       ROUND(100.0 * AVG(readmitted), 2) AS readmit_rate_pct
FROM patients
GROUP BY followup_7d
"""))

# 3.3 Readmission by number of prior admissions
print(q("""
SELECT CASE WHEN prior_admissions >= 4 THEN '4+' ELSE CAST(prior_admissions AS TEXT) END AS prior_adm,
       COUNT(*) AS patients,
       ROUND(100.0 * AVG(readmitted), 2) AS readmit_rate_pct
FROM patients
GROUP BY prior_adm
ORDER BY prior_adm
"""))

In [ ]:
# 3.4 High-risk cohort: 3+ prior admissions, 3+ comorbidities, no follow-up
print(q("""
SELECT COUNT(*) AS patients,
       ROUND(100.0 * AVG(readmitted), 2) AS readmit_rate_pct
FROM patients
WHERE prior_admissions >= 3 AND comorbidity_count >= 3 AND followup_7d = 0
"""))

# 3.5 Readmission by diagnosis and insurance (window function: rank within diagnosis)
print(q("""
WITH t AS (
    SELECT diagnosis, insurance, COUNT(*) AS patients,
           ROUND(100.0 * AVG(readmitted), 2) AS readmit_rate_pct
    FROM patients GROUP BY diagnosis, insurance HAVING COUNT(*) >= 30
)
SELECT diagnosis, insurance, patients, readmit_rate_pct,
       RANK() OVER (PARTITION BY diagnosis ORDER BY readmit_rate_pct DESC) AS rank_in_diagnosis
FROM t
ORDER BY diagnosis, rank_in_diagnosis
"""))

## 4. Exploratory analysis (Matplotlib)

In [ ]:
def rate_bar(col, title, fname, color="tab:blue"):
    r = df.groupby(col, observed=True)["readmitted"].mean() * 100
    fig, ax = plt.subplots()
    ax.bar(r.index.astype(str), r.values, color=color)
    ax.axhline(df["readmitted"].mean() * 100, color="black", linestyle="--", label="Overall rate")
    ax.set_title(title)
    ax.set_ylabel("30-day readmission rate (%)")
    ax.legend()
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(f"{OUT}/{fname}", dpi=150)
    plt.show()

rate_bar("diagnosis", "Readmission Rate by Diagnosis", "rate_by_diagnosis.png")
rate_bar("age_group", "Readmission Rate by Age Group", "rate_by_age.png", "tab:orange")
rate_bar("followup_label", "Readmission Rate: Follow-up vs No Follow-up", "rate_by_followup.png", "tab:green")
rate_bar("prior_group", "Readmission Rate by Prior Admissions", "rate_by_prior.png", "tab:red")

In [ ]:
fig, ax = plt.subplots()
ax.hist(df.loc[df["readmitted"] == 0, "length_of_stay"], bins=20, alpha=0.6,
        density=True, label="Not readmitted")
ax.hist(df.loc[df["readmitted"] == 1, "length_of_stay"], bins=20, alpha=0.6,
        density=True, label="Readmitted")
ax.set_title("Length of Stay: Readmitted vs Not")
ax.set_xlabel("Days")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/los_distribution.png", dpi=150)
plt.show()

## 5. Correlation analysis and statistical tests
Correlation with a 0/1 outcome (readmitted) is called point-biserial correlation. Pearson's `r`
gives the same number. Closer to +1 means "more of this goes with more readmission".

In [ ]:
numeric = ["age", "prior_admissions", "comorbidity_count", "length_of_stay",
           "num_medications", "num_lab_procedures", "bmi", "systolic_bp", "followup_7d"]

corr_rows = []
for c in numeric:
    r, p = stats.pearsonr(df[c], df["readmitted"])
    corr_rows.append({"factor": c, "correlation_with_readmission": r, "p_value": p})
corr_df = pd.DataFrame(corr_rows).sort_values("correlation_with_readmission", ascending=False)
print(corr_df.to_string(index=False))

fig, ax = plt.subplots()
cd = corr_df.sort_values("correlation_with_readmission")
ax.barh(cd["factor"], cd["correlation_with_readmission"],
        color=["tab:red" if v > 0 else "tab:green" for v in cd["correlation_with_readmission"]])
ax.axvline(0, color="black")
ax.set_title("Correlation of Each Factor with 30-day Readmission")
plt.tight_layout()
plt.savefig(f"{OUT}/correlation_with_readmission.png", dpi=150)
plt.show()

In [ ]:
cm = df[numeric + ["readmitted"]].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm)), cm.columns, rotation=60, ha="right")
ax.set_yticks(range(len(cm)), cm.columns)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax.text(j, i, f"{cm.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Correlation Matrix")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(f"{OUT}/correlation_matrix.png", dpi=150)
plt.show()

In [ ]:
# Chi-square tests for categorical factors (is the factor related to readmission?)
chi_rows = []
for c in ["diagnosis", "insurance", "discharge_disposition", "followup_7d", "gender"]:
    table = pd.crosstab(df[c], df["readmitted"])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    cramers_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
    chi_rows.append({"factor": c, "chi2": chi2, "p_value": p, "cramers_v": cramers_v})
chi_df = pd.DataFrame(chi_rows).sort_values("cramers_v", ascending=False)
print(chi_df.to_string(index=False))

# t-test: length of stay
a = df.loc[df["readmitted"] == 1, "length_of_stay"]
b = df.loc[df["readmitted"] == 0, "length_of_stay"]
t, p = stats.ttest_ind(a, b, equal_var=False)
print(f"\nt-test length of stay: readmitted {a.mean():.2f} vs not {b.mean():.2f} days "
      f"(t = {t:.2f}, p = {p:.4g})")

## 6. Logistic regression: which factors raise the risk, holding the others fixed?
Correlation looks at one factor at a time. Logistic regression looks at all factors together.
**Odds ratio > 1** means higher readmission risk, **< 1** means lower risk.

In [ ]:
formula = ("readmitted ~ age + prior_admissions + comorbidity_count + length_of_stay "
           "+ num_medications + followup_7d "
           "+ C(diagnosis, Treatment('Hypertension')) "
           "+ C(discharge_disposition, Treatment('Home')) "
           "+ C(insurance, Treatment('Private')) + C(gender)")

# Train/test split (80/20) to check the model on patients it has not seen
perm = rng.permutation(len(df))
cut = int(0.8 * len(df))
train, test = df.iloc[perm[:cut]], df.iloc[perm[cut:]]
model_train = smf.logit(formula, data=train).fit(disp=0)

def auc_score(y, p):
    ranks = stats.rankdata(p)
    n1 = int(y.sum())
    n0 = len(y) - n1
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

auc = auc_score(test["readmitted"].values, model_train.predict(test).values)
print(f"AUC on unseen test patients: {auc:.3f}  (0.5 = guessing, 1.0 = perfect)")

# Final model on all data (used for the odds ratios and the intervention estimates)
model = smf.logit(formula, data=df).fit(disp=0)

import re
def nice(name):
    m = re.match(r"C\((\w+), Treatment\('([^']+)'\)\)\[T\.(.+)\]", name)
    if m:
        return f"{m.group(1)}: {m.group(3)} (vs {m.group(2)})"
    m = re.match(r"C\((\w+)\)\[T\.(.+)\]", name)
    if m:
        return f"{m.group(1)}: {m.group(2)}"
    return name

ci = model.conf_int()
or_df = pd.DataFrame({
    "factor": [nice(n) for n in model.params.index],
    "odds_ratio": np.exp(model.params.values),
    "ci_low": np.exp(ci[0].values),
    "ci_high": np.exp(ci[1].values),
    "p_value": model.pvalues.values,
}).query("factor != 'Intercept'").sort_values("odds_ratio", ascending=False)
or_df["significant"] = or_df["p_value"] < 0.05
print(or_df.to_string(index=False))

In [ ]:
plot_df = or_df.sort_values("odds_ratio")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_df["factor"], plot_df["odds_ratio"] - 1, left=1,
        color=["tab:red" if v > 1 else "tab:green" for v in plot_df["odds_ratio"]])
ax.axvline(1, color="black")
ax.set_title("Odds Ratios for 30-day Readmission (1 = no effect)")
plt.tight_layout()
plt.savefig(f"{OUT}/odds_ratios.png", dpi=150)
plt.show()

## 7. Intervention estimates: how much could readmission fall?
We use the fitted model to ask "what if" questions. We change `followup_7d` to 1 for some
patients and see how the predicted readmission rate changes.

**Important:** this assumes the follow-up effect found by the model is a real cause. Real
hospitals would confirm this with a pilot or trial. Treat these as estimates, not guarantees.

In [ ]:
base_p = model.predict(df)
base_rate = base_p.mean()
print(f"Observed readmission rate: {df['readmitted'].mean():.2%}")
print(f"Model baseline rate:       {base_rate:.2%}\n")

def scenario(mask):
    d2 = df.copy()
    d2.loc[mask, "followup_7d"] = 1
    new_rate = model.predict(d2).mean()
    return new_rate

# A: everyone gets a follow-up within 7 days
rate_a = scenario(np.ones(len(df), dtype=bool))

# B: only the 20% highest-risk patients who currently have no follow-up
thr = base_p.quantile(0.80)
mask_b = ((base_p >= thr) & (df["followup_7d"] == 0)).values
rate_b = scenario(mask_b)

# C: same as B, but only half of those patients actually attend (realistic uptake)
mask_c = mask_b & (rng.random(len(df)) < 0.5)
rate_c = scenario(mask_c)

rows = []
for name, rate, n_treated in [
    ("A: follow-up for everyone", rate_a, int((df["followup_7d"] == 0).sum())),
    ("B: follow-up for top 20% high-risk", rate_b, int(mask_b.sum())),
    ("C: same as B, 50% attend", rate_c, int(mask_c.sum())),
]:
    rows.append({
        "scenario": name,
        "patients_reached": n_treated,
        "new_readmit_rate": rate * 100,
        "relative_reduction_pct": (base_rate - rate) / base_rate * 100,
        "readmissions_avoided": (base_rate - rate) * len(df),
    })
interv = pd.DataFrame(rows)
print(interv.to_string(index=False))

# What the raw data says, without a model
obs_f = df.groupby("followup_7d")["readmitted"].mean()
print(f"\nObserved: no follow-up {obs_f[0]:.2%} vs follow-up {obs_f[1]:.2%} "
      f"({(obs_f[0] - obs_f[1]) / obs_f[0]:.1%} relative difference)")

## 8. Interactive dashboard with filters (this is the UI)
Use the filter boxes (Ctrl/Cmd + click to select several). KPI cards and charts update.
Works in Google Colab and in VS Code notebooks.

In [ ]:
try:
    import plotly, ipywidgets
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly", "ipywidgets"])

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

diag_opts = sorted(df["diagnosis"].unique())
age_opts = [str(x) for x in df["age_group"].cat.categories]
ins_opts = sorted(df["insurance"].unique())

w_diag = widgets.SelectMultiple(options=diag_opts, value=tuple(diag_opts), description="Diagnosis",
                                rows=6, layout=widgets.Layout(width="260px"))
w_age = widgets.SelectMultiple(options=age_opts, value=tuple(age_opts), description="Age group",
                               rows=5, layout=widgets.Layout(width="230px"))
w_ins = widgets.SelectMultiple(options=ins_opts, value=tuple(ins_opts), description="Insurance",
                               rows=4, layout=widgets.Layout(width="240px"))
dash_out = widgets.Output()

def kpi_card(title, value):
    return (f"<div style='flex:1;min-width:150px;padding:14px;margin:4px;border-radius:10px;"
            f"background:#f1f5f9;border:1px solid #cbd5e1;text-align:center;color:#0f172a'>"
            f"<div style='font-size:12px;color:#475569'>{title}</div>"
            f"<div style='font-size:22px;font-weight:700'>{value}</div></div>")

def draw_dashboard(_=None):
    with dash_out:
        clear_output(wait=True)
        d = df[df["diagnosis"].isin(w_diag.value)
               & df["age_group"].astype(str).isin(w_age.value)
               & df["insurance"].isin(w_ins.value)]
        if d.empty:
            print("No patients for this selection. Pick at least one option in each filter.")
            return
        cards = "".join([
            kpi_card("Patients", f"{len(d):,}"),
            kpi_card("Readmission Rate", f"{d['readmitted'].mean():.1%}"),
            kpi_card("Avg Length of Stay", f"{d['length_of_stay'].mean():.1f} days"),
            kpi_card("Avg Age", f"{d['age'].mean():.0f}"),
            kpi_card("7-day Follow-up", f"{d['followup_7d'].mean():.0%}"),
            kpi_card("Avg Prior Admissions", f"{d['prior_admissions'].mean():.2f}"),
        ])
        display(HTML(f"<div style='display:flex;flex-wrap:wrap'>{cards}</div>"))

        by_dx = d.groupby("diagnosis")["readmitted"].mean().sort_values(ascending=False) * 100
        by_age = d.groupby("age_group", observed=True)["readmitted"].mean() * 100
        by_fu = d.groupby("followup_label")["readmitted"].mean() * 100
        by_pr = d.groupby("prior_group", observed=True)["readmitted"].mean() * 100

        fig = make_subplots(rows=2, cols=2, subplot_titles=(
            "Readmission % by Diagnosis", "Readmission % by Age Group",
            "Readmission %: Follow-up vs None", "Readmission % by Prior Admissions"))
        fig.add_trace(go.Bar(x=by_dx.index, y=by_dx.values, marker_color="#2563eb"), 1, 1)
        fig.add_trace(go.Bar(x=by_age.index.astype(str), y=by_age.values, marker_color="#f97316"), 1, 2)
        fig.add_trace(go.Bar(x=by_fu.index, y=by_fu.values, marker_color="#16a34a"), 2, 1)
        fig.add_trace(go.Scatter(x=by_pr.index.astype(str), y=by_pr.values, mode="lines+markers",
                                 line=dict(color="#dc2626")), 2, 2)
        fig.update_layout(height=700, showlegend=False, margin=dict(t=60, b=20))
        fig.update_yaxes(title_text="%")
        fig.show()

for w in (w_diag, w_age, w_ins):
    w.observe(draw_dashboard, names="value")

display(HTML("<h2 style='margin:6px 0'>Healthcare Patient Outcome Dashboard</h2>"
             "<div style='color:#475569'>Ctrl/Cmd + click to select several options in a filter.</div>"))
display(widgets.HBox([w_diag, w_age, w_ins]))
display(dash_out)
draw_dashboard()

## 9. Excel validation and Power BI export

In [ ]:
excel_path = f"{OUT}/healthcare_analysis.xlsx"
with pd.ExcelWriter(excel_path) as xw:
    df.to_excel(xw, sheet_name="data", index=False)
    df.pivot_table(index="diagnosis", columns="age_group", values="readmitted",
                   aggfunc="mean", observed=True).to_excel(xw, sheet_name="rate_diagnosis_age")
    df.pivot_table(index="insurance", columns="followup_label", values="readmitted",
                   aggfunc="mean").to_excel(xw, sheet_name="rate_insurance_followup")
    or_df.to_excel(xw, sheet_name="odds_ratios", index=False)
    corr_df.to_excel(xw, sheet_name="correlations", index=False)
    interv.to_excel(xw, sheet_name="interventions", index=False)

# Cross-check: SQL result must equal the Pandas result
sql_rate = q("SELECT AVG(readmitted) AS r FROM patients")["r"][0]
print(f"SQL rate = {sql_rate:.4%} | Pandas rate = {df['readmitted'].mean():.4%} | "
      f"match = {np.isclose(sql_rate, df['readmitted'].mean())}")

df["risk_score"] = base_p.round(4)
df.assign(admission_date=df["admission_date"].dt.date,
          age_group=df["age_group"].astype(str), los_group=df["los_group"].astype(str),
          prior_group=df["prior_group"].astype(str)).to_csv(f"{OUT}/patients_for_powerbi.csv", index=False)

dax = """
-- Power BI: Home > Get Data > Text/CSV > patients_for_powerbi.csv > Load
-- Then Modeling > New measure, and paste each measure below.

Patients = DISTINCTCOUNT(patients[patient_id])
Readmissions = SUM(patients[readmitted])
Readmission Rate = DIVIDE([Readmissions], [Patients])
Avg Length of Stay = AVERAGE(patients[length_of_stay])
Avg Age = AVERAGE(patients[age])
Follow-up Rate = AVERAGE(patients[followup_7d])
Avg Risk Score = AVERAGE(patients[risk_score])

Readmission Rate With Follow-up =
CALCULATE([Readmission Rate], patients[followup_7d] = 1)

Readmission Rate No Follow-up =
CALCULATE([Readmission Rate], patients[followup_7d] = 0)

Follow-up Effect (points) =
[Readmission Rate No Follow-up] - [Readmission Rate With Follow-up]

High Risk Patients =
CALCULATE([Patients], patients[risk_score] >= 0.25)

-- Date table (Modeling > New table), mark as date table, relate to patients[admission_date]:
Date = CALENDAR(DATE(2023,1,1), DATE(2025,12,31))

-- Suggested dashboard layout:
--  Cards: Patients, Readmission Rate, Avg Length of Stay, Follow-up Rate
--  Bar: Readmission Rate by diagnosis; Column: by age_group; Line: by prior_group
--  Clustered bar: Readmission Rate With vs No Follow-up by diagnosis
--  Table: top high-risk patients (risk_score)
--  Slicers: diagnosis, age_group, insurance, admission year
"""
with open(f"{OUT}/powerbi_dax_measures.txt", "w") as f:
    f.write(dax)
print("Files:", sorted(os.listdir(OUT)))

## 10. Findings summary (auto-generated from the results above)

In [ ]:
top_risk = or_df[or_df["significant"] & (or_df["odds_ratio"] > 1)].head(4)
protective = or_df[or_df["significant"] & (or_df["odds_ratio"] < 1)].head(3)
dx_rates = (df.groupby("diagnosis")["readmitted"].mean() * 100).sort_values(ascending=False)
print(f"""
SUMMARY
- {len(df):,} discharges analysed; overall 30-day readmission rate {df['readmitted'].mean():.1%}.
- Highest-risk diagnoses by raw rate: {', '.join(f'{k} ({v:.1f}%)' for k, v in dx_rates.head(3).items())}.
- Strongest risk factors (odds ratio, p < 0.05): {', '.join(f'{r.factor} ({r.odds_ratio:.2f})' for r in top_risk.itertuples())}.
- Factors linked to lower risk: {', '.join(f'{r.factor} ({r.odds_ratio:.2f})' for r in protective.itertuples())}.
- Model quality: AUC {auc:.2f} on unseen patients.
- Estimated impact (model-based):
    A: follow-up for everyone                -> {interv.loc[0, 'relative_reduction_pct']:.1f}% fewer readmissions
    B: follow-up for top 20% high-risk       -> {interv.loc[1, 'relative_reduction_pct']:.1f}% fewer readmissions
    C: B with only 50% attendance            -> {interv.loc[2, 'relative_reduction_pct']:.1f}% fewer readmissions
- Interventions are estimates from a model on synthetic data, not clinical proof.
""")